# Feature Engineering


In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

df = pd.read_parquet("../data/processed/fraud_ecommerce.parquet")

# Drop IP range boundary columns 
df = df.drop(columns=[c for c in ["lower_bound_ip_address", "upper_bound_ip_address"] if c in df.columns])

# keep geolocation missingness as signal
if "country" in df.columns:
    df["geo_missing"] = df["country"].isna().astype(int)
    df["country"] = df["country"].fillna("Unknown")

target = "class"
X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=["number"]).columns.tolist()

num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    remainder="drop"
)

print("Before SMOTE (train):")
print(y_train.value_counts())

X_train_t = preprocess.fit_transform(X_train)

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train_t, y_train)

print("\nAfter SMOTE (train):")
print(pd.Series(y_res).value_counts())


Before SMOTE (train):
class
0    109568
1     11321
Name: count, dtype: int64

After SMOTE (train):
class
0    109568
1    109568
Name: count, dtype: int64
